In [19]:
import cv2
import csv
import re
import os
from ultralytics import YOLO

In [20]:
##yolo11s, epochs=70
#
#def processar_video_yolo(video_path, modelo_path="my_model.pt"):
#    """
#    Processa um vídeo com YOLO, gera um novo vídeo com detecções e salva um CSV com as coordenadas do objeto detectado.
#
#    Args:
#        video_path (str): Nome do vídeo no formato "solution X - video Y - HD.mp4".
#        modelo_path (str): Caminho para o modelo YOLO treinado (.pt).
#    """
#
#    # **Regex para extrair "X" e "Y" do nome do vídeo**
#    match = re.search(r"solution (\d+) - video (\d+) - HD\.mp4", video_path)
#    if not match:
#        raise ValueError("O nome do vídeo deve estar no formato 'solution X - video Y - HD.mp4'.")
#
#    sol_num, vid_num = match.groups()  # Pega os números X e Y
#    output_video_path = f"sol{sol_num} - v{vid_num} - yolo.mp4"
#    csv_file_path = f"sol{sol_num} - v{vid_num} - yolo.csv"
#
#    print(f"🎬 Processando {video_path}...")
#    print(f"📁 Salvando vídeo em: {output_video_path}")
#    print(f"📁 Salvando CSV em: {csv_file_path}")
#
#    # Carregar o modelo YOLO
#    model = YOLO(modelo_path)
#
#    # Abrir o vídeo
#    cap = cv2.VideoCapture(video_path)
#
#    # Pegar FPS e tamanho do vídeo original
#    fps = int(cap.get(cv2.CAP_PROP_FPS))
#    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#
#    # Criar writer para salvar o vídeo processado
#    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
#    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
#
#    # Criar CSV e escrever cabeçalho
#    with open(csv_file_path, mode="w", newline="") as file:
#        writer = csv.writer(file)
#        writer.writerow(["Frame", "Confiança", "Centro_X", "Centro_Y", "Diâmetro"])
#
#        frame_count = 0  # Contador de frames
#
#        while cap.isOpened():
#            ret, frame = cap.read()
#            if not ret:
#                break  # Sai do loop se o vídeo terminou
#
#            frame_count += 1  # Incrementa o contador de frames
#
#            # Rodar YOLO no frame
#            results = model(frame)
#
#            # Processar detecções
#            for r in results:
#                for box in r.boxes:
#                    x1, y1, x2, y2 = map(int, box.xyxy[0])
#                    conf = float(box.conf[0])
#
#                    # Calcular centro da detecção
#                    center_x = (x1 + x2) // 2
#                    center_y = (y1 + y2) // 2
#
#                    # Estimar diâmetro (considerando um objeto aproximadamente esférico)
#                    diameter = max((x2 - x1), (y2 - y1))
#
#                    # Salvar dados no CSV
#                    writer.writerow([frame_count, conf, center_x, center_y, diameter])
#
#                    # Desenhar a detecção no frame
#                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#                    cv2.circle(frame, (center_x, center_y), 3, (0, 0, 255), -1)
#                    cv2.putText(frame, f"{conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
#
#            # Escrever frame processado no vídeo
#            out.write(frame)
#
#    # Liberar recursos
#    cap.release()
#    out.release()
#
#    print(f"✅ Processamento concluído! Vídeo salvo como: {output_video_path}")
#    print(f"✅ CSV salvo como: {csv_file_path}")

In [21]:
def processar_video_yolo(video_path, modelo_path="my_model.pt"):
    """
    Processa um vídeo com YOLO, gera um novo vídeo com detecções e salva um CSV com as coordenadas do objeto detectado.

    Args:
        video_path (str): Caminho completo do vídeo, por exemplo:
                          r"D:\ClawLike Tweezer to Biology\reproduction\video 1 - reproduction\video 1 - reproduction.mp4"
        modelo_path (str): Caminho para o modelo YOLO treinado (.pt).
    """

    # Extrair nome da pasta e do arquivo para identificar os números
    folder_name = os.path.basename(os.path.dirname(video_path))
    file_name = os.path.basename(video_path)

    # Buscar número do vídeo dentro do nome do arquivo ou da pasta
    match = re.search(r"video (\d+)", folder_name + " " + file_name)
    if not match:
        raise ValueError("Não foi possível extrair o número do vídeo. Certifique-se de que o nome contém 'video X'.")

    vid_num = match.group(1)
    output_video_path = f"video{vid_num} - yolo.mp4"
    csv_file_path = f"video{vid_num} - yolo.csv"

    print(f"🎬 Processando {video_path}...")
    print(f"📁 Salvando vídeo em: {output_video_path}")
    print(f"📁 Salvando CSV em: {csv_file_path}")

    # Carregar modelo YOLO
    model = YOLO(modelo_path)

    # Abrir vídeo
    cap = cv2.VideoCapture(video_path)

    # FPS e dimensões
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Criar writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # Criar CSV
    with open(csv_file_path, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Frame", "Confiança", "Centro_X", "Centro_Y", "Diâmetro"])

        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            results = model(frame)

            for r in results:
                for box in r.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    center_x = (x1 + x2) // 2
                    center_y = (y1 + y2) // 2
                    diameter = max((x2 - x1), (y2 - y1))

                    writer.writerow([frame_count, conf, center_x, center_y, diameter])

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.circle(frame, (center_x, center_y), 3, (0, 0, 255), -1)
                    cv2.putText(frame, f"{conf:.2f}", (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            out.write(frame)

    cap.release()
    out.release()

    print(f"✅ Processamento concluído! Vídeo salvo como: {output_video_path}")
    print(f"✅ CSV salvo como: {csv_file_path}")

In [22]:
processar_video_yolo(r"/Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 1 - reproduction/control group - video 1 - reproduction.mp4", modelo_path="my_model.pt")

🎬 Processando /Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 1 - reproduction/control group - video 1 - reproduction.mp4...
📁 Salvando vídeo em: video1 - yolo.mp4
📁 Salvando CSV em: video1 - yolo.csv

0: 384x640 8 Saccharomyces Cerevisiaes, 80.5ms
Speed: 7.8ms preprocess, 80.5ms inference, 9.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 Saccharomyces Cerevisiaes, 63.8ms
Speed: 1.0ms preprocess, 63.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Saccharomyces Cerevisiaes, 63.7ms
Speed: 0.9ms preprocess, 63.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Saccharomyces Cerevisiaes, 62.4ms
Speed: 1.4ms preprocess, 62.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Saccharomyces Cerevisiaes, 64.1ms
Speed: 1.1ms preprocess, 64.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Saccharomyces Cerevi

In [23]:
processar_video_yolo(r"/Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 2 - reproduction/control group - video 2 - reproduction.mp4", modelo_path="my_model.pt")

🎬 Processando /Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 2 - reproduction/control group - video 2 - reproduction.mp4...
📁 Salvando vídeo em: video2 - yolo.mp4
📁 Salvando CSV em: video2 - yolo.csv

0: 384x640 4 Saccharomyces Cerevisiaes, 66.4ms
Speed: 0.9ms preprocess, 66.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Saccharomyces Cerevisiaes, 64.2ms
Speed: 1.0ms preprocess, 64.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Saccharomyces Cerevisiaes, 66.2ms
Speed: 1.0ms preprocess, 66.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Saccharomyces Cerevisiaes, 66.4ms
Speed: 0.9ms preprocess, 66.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Saccharomyces Cerevisiaes, 64.3ms
Speed: 1.0ms preprocess, 64.3ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 Saccharomyces Cerevi

In [27]:
processar_video_yolo(r"/Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 3 - reproduction/control group - video 3 - reproduction.mp4", modelo_path="my_model.pt")

🎬 Processando /Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 3 - reproduction/control group - video 3 - reproduction.mp4...
📁 Salvando vídeo em: video3 - yolo.mp4
📁 Salvando CSV em: video3 - yolo.csv

0: 384x640 2 Saccharomyces Cerevisiaes, 68.1ms
Speed: 1.1ms preprocess, 68.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 75.1ms
Speed: 21.6ms preprocess, 75.1ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 68.8ms
Speed: 1.1ms preprocess, 68.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 67.4ms
Speed: 1.4ms preprocess, 67.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 66.2ms
Speed: 1.0ms preprocess, 66.2ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerev

In [25]:
processar_video_yolo(r"/Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 4 - reproduction/control group - video 4 - reproduction.mp4", modelo_path="my_model.pt")

🎬 Processando /Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 4 - reproduction/control group - video 4 - reproduction.mp4...
📁 Salvando vídeo em: video4 - yolo.mp4
📁 Salvando CSV em: video4 - yolo.csv

0: 384x640 2 Saccharomyces Cerevisiaes, 74.6ms
Speed: 1.0ms preprocess, 74.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Saccharomyces Cerevisiae, 68.4ms
Speed: 1.1ms preprocess, 68.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Saccharomyces Cerevisiae, 65.4ms
Speed: 1.1ms preprocess, 65.4ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 68.7ms
Speed: 1.1ms preprocess, 68.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 71.5ms
Speed: 1.1ms preprocess, 71.5ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisi

In [26]:
processar_video_yolo(r"/Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 5 - reproduction/control group - video 5 - reproduction.mp4", modelo_path="my_model.pt")

🎬 Processando /Users/arielhertz/Desktop/Claw-like Tweezer/reprodutcion - control group/control, video 5 - reproduction/control group - video 5 - reproduction.mp4...
📁 Salvando vídeo em: video5 - yolo.mp4
📁 Salvando CSV em: video5 - yolo.csv

0: 384x640 2 Saccharomyces Cerevisiaes, 69.9ms
Speed: 1.1ms preprocess, 69.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 64.6ms
Speed: 1.1ms preprocess, 64.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 64.9ms
Speed: 1.0ms preprocess, 64.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 Saccharomyces Cerevisiaes, 67.8ms
Speed: 0.9ms preprocess, 67.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Saccharomyces Cerevisiae, 67.2ms
Speed: 1.0ms preprocess, 67.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Saccharomyces Cerevis